# Preparación y control de calidad de datos

**Objetivos**
Estandarización de fechas, nombres de variables, unidades y formatos.
Detección de duplicados, valores imposibles y valores atípicos.
Diagnóstico de huecos y regularidad temporal.
Interpolación o imputación cuando esté justificada, conservando los datos originales y marcando los valores modificados.
Registro de todas las transformaciones.

**Datasets utilizados** Entrada: datos brutos (raw/) · Salida: conjunto de datos depurado y documentado (processed/).

*Datasets hidrología*: capturados con CTD y sensores acoplados a la roseta. Tras el procesamiento, se generan datos en formas de perfiles verticales con resolución espacial de 1m. En este ejemplo, estos son los datos de temperatura, salinidad, fluorescencia y oxígeno. (Format data as df for SARIMAX).

In [10]:
# Select hidro data
datos<-read.csv("../data/raw/Hidro_E2GI_insitu.csv")
#dim(datos)
#colnames(datos)
hidro_surface<-datos[datos$Press==5,]
#colnames(hidro_surface)
# Excluir concentraciones ausentes del cálculo de la media
validos <- !is.na(hidro_surface$Fluor)
datos <- hidro_surface[validos, ]

In [7]:
# Crear una fecha correspondiente al inicio de cada mes
fechas3<-as.Date(paste(paste(datos$YYYY,sprintf("%02d",datos$MM),sprintf("%02d",datos$DD), sep="-")))
datos$mes <- as.Date(paste(paste(datos$YYYY,sprintf("%02d",datos$MM),sprintf("%02d",rep(1, length(datos$DD))), sep="-")))

# Agregar por mes
meses <- sort(unique(datos$mes))
hidro_m <- data.frame(mes = meses,temp = rep(NA,length(meses)), sal = rep(NA,length(meses)),
                      fluor = rep(NA,length(meses)), o2 = rep(NA,length(meses)), n_muestras = integer(length(meses)))

for (i in seq_along(meses)) {
  valores <- datos$Temp[datos$mes == meses[i]]
  hidro_m$temp[i] <- mean(valores)
  valores <- datos$Sal[datos$mes == meses[i]]
  hidro_m$sal[i] <- mean(valores)
  valores <- datos$Fluor[datos$mes == meses[i]]
  hidro_m$fluor[i] <- mean(valores)
  valores <- datos$O2[datos$mes == meses[i]]
  hidro_m$o2[i] <- mean(valores)
  hidro_m$n_muestras[i] <- length(valores)
}

# Mostrar el resultado
head(hidro_m, 12)

,mes,temp,sal,fluor,o2,n_muestras
,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,2001-04-01,13.5672,34.8607,0.1333400,NA,1
2,2001-06-01,16.2020,35.1021,0.0589480,NA,1
3,2001-07-01,19.2614,35.2206,0.0337870,NA,1
4,2001-08-01,20.8440,35.1640,0.0363130,NA,1
5,2001-09-01,19.3269,34.8126,0.0608610,NA,1
6,2001-10-01,17.4475,35.2823,0.1051600,NA,1
7,2002-01-01,13.0699,35.7381,0.0622710,NA,1
8,2002-02-01,13.2559,35.7976,0.0195360,NA,1
9,2002-03-01,13.0708,35.6304,0.0368970,NA,1


In [11]:
# Incorporar los meses sin observaciones
# Calendario mensual completo
calendario <- seq(from = min(hidro_m$mes),to = max(hidro_m$mes),by = "month")
indice <- match(calendario, hidro_m$mes)

df <- data.frame(
  mes = calendario,
  temp = hidro_m$temp[indice],
  sal = hidro_m$sal[indice],
  fluor = hidro_m$fluor[indice],
  o2 = hidro_m$o2[indice],
  n_muestras = hidro_m$n_muestras[indice])

# Resumen de cobertura
cat("Meses totales:", nrow(df), "\n")
cat("Meses con observaciones:", sum(!is.na(df$chla)), "\n")
cat("Meses sin observaciones:", sum(is.na(df$chla)), "\n")

cat(
  "Porcentaje de meses observados:",
  round(100 * mean(!is.na(df$chla)), 1),
  "%\n")

Meses totales: 256 
Meses con observaciones: 0 
Meses sin observaciones: 0 
Porcentaje de meses observados: NaN %


In [12]:
# Guardar df como CSV
write.csv(df,
  "../data/processed/Hidro_E2GI_surface_monthly.csv",
  row.names = FALSE,na = "")
colnames(df)

[1] "mes"        "temp"       "sal"        "fluor"      "o2"        
[6] "n_muestras"

*Datasets botellas*: medidos en el agua recogida con botellas acopladas a la roseta oceanográfica. Los datos forman perfiles con datos a profundidades discretas. En este ejemplo, estos son los datos de nutrientes y clorofila.

In [13]:
datos<-read.csv("../data/raw/Chla_E2GI_insitu.csv")
#dim(datos)
#colnames(datos)
chla_surface<-datos[datos$Depth==0 & datos$Fraction=="GFF",]
#colnames(chla_surface)

# Excluir concentraciones ausentes del cálculo de la media
validos <- !is.na(chla_surface$Chl.a)
datos <- chla_surface[validos, ]

In [14]:
# Crear una fecha correspondiente al inicio de cada mes
#fechas3<-as.Date(paste(paste(datos$YYYY,sprintf("%02d",datos$MM),sprintf("%02d",datos$DD), sep="-")))
datos$mes <- as.Date(paste(paste(datos$YYYY,sprintf("%02d",datos$MM),sprintf("%02d",rep(1, length(datos$DD))), sep="-")))
# Agregar por mes
meses <- sort(unique(datos$mes))
chla_m <- data.frame(mes = meses,chla = NA_real_,n_muestras = integer(length(meses)))

for (i in seq_along(meses)) {
  valores <- datos$Chl.a[datos$mes == meses[i]]
  chla_m$chla[i] <- mean(valores)
  chla_m$n_muestras[i] <- length(valores)
}

# Mostrar el resultado
head(chla_m, 12)

,mes,chla,n_muestras
,<date>,<dbl>,<int>
1,2001-05-01,0.801,1
2,2001-06-01,0.211,1
3,2001-07-01,0.127,1
4,2001-08-01,0.197,1
5,2001-09-01,0.107,1
6,2001-10-01,0.589,1
7,2001-11-01,1.121,1
8,2001-12-01,0.848,1
9,2002-01-01,0.674,1


In [15]:
# Incorporar los meses sin observaciones
# Calendario mensual completo
calendario <- seq(from = min(chla_m$mes),to = max(chla_m$mes),by = "month")
indice <- match(calendario, chla_m$mes)

df <- data.frame(
  mes = calendario,
  chla = chla_m$chla[indice],
  n_muestras = chla_m$n_muestras[indice])

# Resumen de cobertura
cat("Meses totales:", nrow(df), "\n")
cat("Meses con observaciones:", sum(!is.na(df$chla)), "\n")
cat("Meses sin observaciones:", sum(is.na(df$chla)), "\n")

cat("Porcentaje de meses observados:",
  round(100 * mean(!is.na(df$chla)), 1),"%\n")

Meses totales: 260 
Meses con observaciones: 169 
Meses sin observaciones: 91 
Porcentaje de meses observados: 65 %


In [16]:
# Guardar df como CSV
write.csv(df,"../data/processed/Chla_E2GI_surface_monthly.csv",
  row.names = FALSE,na = "")

*Datasets pescas*: medidos en el agua filtrada por redes de plancton arrastradas oblicua o verticalmente. El dato representa toda la profundidad de la pesca. En este ejemplo, estos son los datos de zooplancton.

Format data as input for GOTM-FABM (.dat)
Format data as observations for GOTM-FABM (.obs)